# CoDraft rebuttal runs (ARR October 2026), Kaggle 2×T4

Attach two datasets: **codraft-dataset** (code + data) and **codraft-weight** (released checkpoints of runs 3–7 and every small result file). Then use *Save Version → Save & Run All* so the session runs in the background.

| session | runs | what | ~time on 2×T4 |
|---|---|---|---|
| `"A"` | 30–33, 26–29, 42–47, 34 | E2 seeds, E1 same-recipe CE, E9a, E5 counterfactual, E1b, E6 | ~7 h |
| `"B"` | 35–41 | E9b, E7 CORAL/CORN, E2 seeds of the same-recipe CE | ~6.5 h |
| `"C"` | 48, 49 | E7 CORAL fixed (prior-initialised biases at lr 1e-3 / 1e-2), one per GPU | ~3 h |

The notebook:
1. runs a **smoke test** of every code path in the session (both GPUs; it stops here if anything fails);
2. runs the full session, one run per GPU from a shared queue, skipping any run that would not finish within `BUDGET_HOURS`;
3. zips every result into `/kaggle/working/rebuttal_<SESSION>.zip`. Download that file and run `python runner.py --install rebuttal_<SESSION>.zip` locally.

Model weights of the new runs are **not saved** (`CODRAFT_SAVE_WEIGHTS=0`): only preds, metrics, logits and logs, which is all the rebuttal needs.

In [ ]:
SESSION = "C"        # "A", "B", "C", or a list of run IDs, e.g. [35, 36]
RUN_SMOKE = True     # smoke-test every code path first (~15-20 min); stop if any fails
BUDGET_HOURS = 11.0  # Kaggle stops a session at 12 h and then keeps no output

In [ ]:
import glob
import os
import subprocess
import sys
import time

T0 = time.time()


def sh(*cmd: str) -> None:
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


if os.path.isdir("/kaggle/working"):
    OUT_ROOT = "/kaggle/working"
    REPO_DIR = os.path.dirname(sorted(glob.glob("/kaggle/input/**/runner.py", recursive=True))[0])
    sh(sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt")
else:
    REPO_DIR = os.getcwd()
    while REPO_DIR != "/" and not os.path.isfile(os.path.join(REPO_DIR, "runner.py")):
        REPO_DIR = os.path.dirname(REPO_DIR)
    OUT_ROOT = REPO_DIR

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
os.environ["CODRAFT_SAVE_WEIGHTS"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"])

from config.runs import RUNS, SESSIONS, EST_MIN, needs_checkpoints, smoke_ids
from runner import find_release_dir, launch, run_tag

IDS = SESSIONS[SESSION] if isinstance(SESSION, str) else list(SESSION)
for r in IDS:
    print(f"{r:3d} ~{EST_MIN.get(r, '?'):>4} min  {RUNS[r]['name']:32s} {RUNS[r]['row']}")
missing = [s for s in needs_checkpoints(IDS) if find_release_dir(s) is None]
if missing:
    raise FileNotFoundError(f"Checkpoints of runs {missing} not found: attach the codraft-weight dataset")
for s in needs_checkpoints(IDS):
    print(f"checkpoint of run {s}: {find_release_dir(s)}")
print(f"Code {REPO_DIR} | output {OUT_ROOT} | session {SESSION}: {IDS}")

In [ ]:
import json
import zipfile


def show(smoke: bool, ids: list) -> None:
    for r in ids:
        f = f"{OUT_ROOT}/results/{run_tag(r, smoke)}_metrics.json"
        if not os.path.isfile(f):
            print(f"{r:3d} no result")
            continue
        m = json.load(open(f))
        st = m["settings"]
        extra = []
        if st.get("mem_probe_gb") is not None or "gradient_checkpointing" in st:
            extra.append(f"VRAM probe {st.get('mem_probe_gb')} GiB, GC {'on' if st.get('gradient_checkpointing') else 'off'}")
        if "two_order" in st:
            extra.append(f"two orders {st['two_order']['f1_macro'] * 100:.2f}")
        if "ab_metrics" in st:
            extra.append(f"(A,B) {st['ab_metrics']['f1_macro'] * 100:.2f}")
        for k in ("aux_acc_val", "aux_acc_test"):
            if st.get(k) is not None:
                extra.append(f"{k} {st[k] * 100:.2f}%")
        if "reproduction" in st:
            extra.append(f"agreement with original {st['reproduction']['pred_agreement'] * 100:.2f}%")
        if "conditions" in st and "pairs" in st:
            for name, c in st["conditions"].items():
                if "n" in c:
                    extra.append(f"{name}: {c['wrong_to_right']} w->r / {c['right_to_wrong']} r->w of {c['n']}")
        print(
            f"{r:3d} {m['tag']:44s} F1 {m['f1_macro'] * 100:6.2f} QWK {m['qwk']:.3f} MAE {m['mae']:.3f} "
            f"Sev {m['severe_rate'] * 100:5.2f}% n={m['n_test']} {m['train_seconds'] / 60:6.1f} min"
        )
        for e in extra:
            print("      " + e)


if RUN_SMOKE:
    SMOKE_IDS = smoke_ids(IDS)
    print("Smoke test of", SMOKE_IDS)
    try:
        launch(SMOKE_IDS, out_root=OUT_ROOT, smoke=True, package_zip=False)
    finally:
        show(True, SMOKE_IDS)
        with zipfile.ZipFile(f"{OUT_ROOT}/smoke_logs_{SESSION}.zip", "w", zipfile.ZIP_DEFLATED) as zf:
            for f in glob.glob(f"{OUT_ROOT}/logs/SMOKE_*.log"):
                zf.write(f, os.path.relpath(f, OUT_ROOT))
    print(f"Smoke test passed in {(time.time() - T0) / 60:.0f} min since start")

In [ ]:
FULL = {"failed": [], "skipped": [], "error": None}
budget = BUDGET_HOURS - (time.time() - T0) / 3600
print(f"Full runs {IDS}, {budget:.2f} h left in the budget")
try:
    FULL.update(launch(IDS, out_root=OUT_ROOT, budget_hours=budget, package_zip=False))
except Exception as e:  # keep going so the results that exist are zipped below
    FULL["error"] = f"{type(e).__name__}: {e}"
    print("ERROR:", FULL["error"])
show(False, IDS)

In [ ]:
import glob
import os
import zipfile


def zip_results(out_root: str, session: object) -> str:
    """results/ and logs/ of the full runs (no SMOKE_ files, no confusion-matrix images) in one zip
    that `python runner.py --install` unpacks into weights/."""
    name = session if isinstance(session, str) else "-".join(map(str, session))
    path = os.path.join(out_root, f"rebuttal_{name}.zip")
    files = []
    for sub in ("results", "logs"):
        for f in sorted(glob.glob(os.path.join(out_root, sub, "*"))):
            base = os.path.basename(f)
            if base.startswith(("SMOKE_", "cm_")) or not os.path.isfile(f):
                continue
            files.append(f)
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
        for f in files:
            zf.write(f, os.path.relpath(f, out_root))
    with zipfile.ZipFile(path) as zf:
        bad = zf.testzip()
    if bad is not None:
        raise OSError(f"{path}: damaged member {bad}")
    print(f"{path}: {len(files)} files, {os.path.getsize(path) / 2**20:.1f} MB")
    return path


# --- run ---
ZIP = zip_results(OUT_ROOT, SESSION)
with zipfile.ZipFile(ZIP) as zf:
    tags = sorted({n.split("/")[-1].rsplit("_metrics.json", 1)[0] for n in zf.namelist() if n.endswith("_metrics.json")})
print("Finished runs in the zip:", tags)
print(f"Total {(time.time() - T0) / 3600:.2f} h")
if FULL["skipped"]:
    print("SKIPPED for time (run them in another session):", FULL["skipped"])
if FULL["failed"] or FULL["error"]:
    print("FAILED:", FULL["failed"], FULL["error"], "- see logs/ in the zip")
print(f"Download {ZIP} from the Output panel, then locally: python runner.py --install {os.path.basename(ZIP)}")